# Day 4 · 02 · GroupBy

`groupby` is pandas' version of SQL `GROUP BY`, and more: besides collapsing groups it can compute per-group values **for every row** and filter whole groups.

**In this notebook**
- Split, apply, combine
- Named aggregation
- `size`, `count` and `nunique`
- `transform`: group results on every row
- Ranking and top N per group
- `filter`: keep or drop whole groups
- Grouping by time with `pd.Grouper`
- Running values inside groups
- Missing group keys
- Custom functions with `apply`

**Dataset:** the retail CSV files, joined into one `sales` frame with one row per order line

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")

sales = (
    items.merge(orders, on="order_id")
    .merge(products[["product_id", "product_name", "category"]], on="product_id")
    .merge(customers[["customer_id", "country", "segment"]], on="customer_id")
)
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount"])

# Cancelled orders are excluded from revenue figures, as in the Day 3 SQL notebooks.
valid = sales[sales["status"] != "cancelled"]

print("all lines:", len(sales), "| lines on non-cancelled orders:", len(valid))
sales.head(3)

all lines: 4439 | lines on non-cancelled orders: 4099


,order_id,line_no,product_id,quantity,unit_price,discount,customer_id,order_date,status,channel,staff_id,product_name,category,country,segment,revenue
0,10001,1,P013,1,38.0,0.0,C0176,2024-01-01,completed,mobile,NaN,Cast Iron Pan,Home,Spain,Small Business,38.0
1,10001,2,P032,4,39.0,0.0,C0176,2024-01-01,completed,mobile,NaN,Pandas in Practice,Books,Spain,Small Business,156.0
2,10002,1,P003,1,39.5,0.0,C0200,2024-01-01,returned,mobile,NaN,USB-C Hub,Electronics,India,Corporate,39.5


## 1. Split, apply, combine
`groupby` works in three steps:
1. **split** the rows into groups by key
2. **apply** a function to each group
3. **combine** the results

`df.groupby("key")["column"].sum()` reads: for each key, the sum of the column.

In [2]:
valid.groupby("category")["revenue"].sum().round(2).sort_values(ascending=False)

category
Electronics    145420.33
Books           70290.25
Sports          65399.62
Home            54476.32
Toys            37648.07
Name: revenue, dtype: float64

## 2. Named aggregation
`.agg(new_name=(column, function))` computes several statistics at once and names the output columns. It is the clearest form and the one to use by default.

These figures match the category query in the Day 3 aggregations notebook. (Floating-point rounding can move the last cent.)

In [3]:
(
    valid.groupby("category")
    .agg(
        orders=("order_id", "nunique"),
        units=("quantity", "sum"),
        revenue=("revenue", "sum"),
        avg_discount=("discount", "mean"),
    )
    .round(2)
    .sort_values("revenue", ascending=False)
)

,orders,units,revenue,avg_discount
category,,,,
Electronics,740,1667,145420.33,0.04
Books,851,1973,70290.25,0.05
Sports,661,1449,65399.62,0.04
Home,739,1644,54476.32,0.05
Toys,505,1130,37648.07,0.04


### Several keys
Pass a list of columns. The result has one row per combination that exists.
- By default the keys become a `MultiIndex`.
- `as_index=False` keeps them as ordinary columns, which is easier to merge and export.

In [4]:
(
    valid.groupby(["country", "segment"], as_index=False)
    .agg(customers=("customer_id", "nunique"), revenue=("revenue", "sum"))
    .round(2)
    .sort_values("revenue", ascending=False)
    .head(5)
)

,country,segment,customers,revenue
9,India,Consumer,21,43256.49
21,USA,Consumer,18,39416.24
0,Canada,Consumer,10,37710.57
6,Germany,Consumer,21,27877.24
4,France,Corporate,11,24137.68


## 3. `size`, `count` and `nunique`
| Function | Counts |
| --- | --- |
| `size` | rows in the group, missing values included |
| `count` | **non-missing** values of a column |
| `nunique` | distinct non-missing values |

`staff_id` is only filled for store orders, so the three differ.

In [5]:
orders.groupby("channel").agg(
    rows=("order_id", "size"),
    with_staff=("staff_id", "count"),
    distinct_staff=("staff_id", "nunique"),
    distinct_customers=("customer_id", "nunique"),
)

,rows,with_staff,distinct_staff,distinct_customers
channel,,,,
mobile,626,0,0,153
store,404,404,8,132
web,970,0,0,175


## 4. `transform`: a group result on every row
`agg` returns one row per group. `transform` returns a result **with the same length as the input**, so it can be stored as a new column. It is the pandas counterpart of a SQL window function with `PARTITION BY`.

Here: each product's share of its category's revenue.

In [6]:
by_product = valid.groupby(["category", "product_name"], as_index=False)["revenue"].sum()

by_product["category_revenue"] = by_product.groupby("category")["revenue"].transform("sum")
by_product["share_pct"] = 100 * by_product["revenue"] / by_product["category_revenue"]

by_product.round(1).head(5)

,category,product_name,revenue,category_revenue,share_pct
0,Books,Charts That Work,7996.5,70290.2,11.4
1,Books,Clean Coding Guide,8595.0,70290.2,12.2
2,Books,Data Science Handbook,5443.2,70290.2,7.7
3,Books,Data Systems Design,17919.2,70290.2,25.5
4,Books,Pandas in Practice,7971.6,70290.2,11.3


## 5. Ranking and top N per group
- `rank` numbers rows inside each group (`method="dense"` behaves like SQL `DENSE_RANK`).
- For "top N per group", sort and take `head(N)` of each group, or filter on the rank.

In [7]:
by_product["position"] = by_product.groupby("category")["revenue"].rank(method="dense", ascending=False).astype(int)

(
    by_product[by_product["position"] <= 2]
    .sort_values(["category", "position"])
    .round(1)[["category", "position", "product_name", "revenue", "share_pct"]]
)

,category,position,product_name,revenue,share_pct
3,Books,1,Data Systems Design,17919.2,25.5
5,Books,2,Python Basics Guide,9443.2,13.4
10,Electronics,1,Noise-Cancelling Headphones,41999.0,28.9
12,Electronics,2,Smart Watch,30949.1,21.3
16,Home,1,Air Fryer,11519.2,21.1
20,Home,2,Memory Foam Pillow,9061.5,16.6
29,Sports,1,Tennis Racket,28438.0,43.5
25,Sports,2,Dumbbell Pair 5kg,9814.5,15.0
36,Toys,1,Remote Control Car,18313.5,48.6
33,Toys,2,Building Blocks 500pc,7106.0,18.9


## 6. `filter`: keep or drop whole groups
`filter` runs a function on each group and keeps **all rows** of the groups where it returns `True`. It is SQL's `HAVING`, except that you get the detail rows back instead of one row per group.

The `transform` version below does the same job without calling a Python function per group, and is the faster habit.

In [8]:
frequent = orders.groupby("customer_id").filter(lambda group: len(group) >= 25)
print(frequent["customer_id"].nunique(), "customers with 25 or more orders,", len(frequent), "orders kept")

order_count = orders.groupby("customer_id")["order_id"].transform("size")
same = orders[order_count >= 25]
print("transform version returns the same rows:", same.equals(frequent))

21 customers with 25 or more orders, 966 orders kept
transform version returns the same rows: True


## 7. Grouping by time
`pd.Grouper(key=..., freq=...)` groups a datetime column into periods without creating a helper column.

| `freq` | Period |
| --- | --- |
| `"D"` | day |
| `"W"` | week |
| `"ME"` | month (labelled by month end) |
| `"QE"` | quarter |
| `"YE"` | year |

In [9]:
monthly = (
    valid.groupby(pd.Grouper(key="order_date", freq="ME"))
    .agg(orders=("order_id", "nunique"), revenue=("revenue", "sum"))
    .round(2)
)
monthly.head(4)

,orders,revenue
order_date,,
2024-01-31,64,12546.68
2024-02-29,71,13498.41
2024-03-31,83,17555.56
2024-04-30,79,14114.22


In [10]:
# Time and another key together, then unstack the category level into columns.
(
    valid.groupby([pd.Grouper(key="order_date", freq="QE"), "category"])["revenue"]
    .sum()
    .unstack("category")
    .round(0)
)

category,Books,Electronics,Home,Sports,Toys
order_date,,,,,
2024-03-31,7970.0,19072.0,6131.0,6391.0,4036.0
2024-06-30,9241.0,22851.0,8457.0,10769.0,5640.0
2024-09-30,9845.0,23133.0,6959.0,10623.0,5277.0
2024-12-31,17080.0,28900.0,10563.0,15922.0,9014.0
2025-03-31,10242.0,22910.0,9912.0,9024.0,6257.0
2025-06-30,15912.0,28554.0,12454.0,12670.0,7424.0


## 8. Running values inside groups
Grouped versions of `cumcount`, `cumsum`, `shift` and `diff` restart for each group. Sort first: they work in row order.

Here: number each customer's orders and measure the gap since their previous one. This is the `LAG` example from the SQL window functions notebook.

In [11]:
history = orders.sort_values(["customer_id", "order_date", "order_id"]).copy()

history["order_number"] = history.groupby("customer_id").cumcount() + 1
history["days_since_previous"] = history.groupby("customer_id")["order_date"].diff().dt.days

history.loc[history["customer_id"] == "C0033", ["customer_id", "order_id", "order_date", "order_number", "days_since_previous"]].head(6)

,customer_id,order_id,order_date,order_number,days_since_previous
13,C0033,10014,2024-01-09,1,NaN
26,C0033,10027,2024-01-15,2,6.0
28,C0033,10029,2024-01-17,3,2.0
35,C0033,10036,2024-01-19,4,2.0
47,C0033,10048,2024-01-24,5,5.0
51,C0033,10052,2024-01-25,6,1.0


In [12]:
# One row per customer from the row-level columns.
(
    history.groupby("customer_id")
    .agg(orders=("order_id", "size"), median_gap_days=("days_since_previous", "median"))
    .sort_values("orders", ascending=False)
    .head(4)
)

,orders,median_gap_days
customer_id,,
C0033,116,3.0
C0181,109,3.0
C0150,73,5.0
C0171,49,7.0


## 9. Missing group keys
Rows whose key is missing are **dropped** from the result by default. Pass `dropna=False` to keep them as their own group. Forgetting this is a common reason for totals that do not add up.

In [13]:
print("rows counted by default      :", orders.groupby("staff_id").size().sum())
print("rows counted with dropna=False:", orders.groupby("staff_id", dropna=False).size().sum())

orders.groupby("staff_id", dropna=False).size().sort_values(ascending=False).head(3)

rows counted by default      : 404
rows counted with dropna=False: 2000


staff_id
NaN    1596
S10      62
S14      60
dtype: int64

## 10. Custom logic with `apply`
`apply` passes each group to your function as a DataFrame. It can return anything, which makes it flexible and **slow**: the function runs once per group in Python.

Reach for it only when `agg`, `transform` and `filter` cannot express the calculation.

In [14]:
def describe_category(group):
    best = group.groupby("product_name")["revenue"].sum()
    return pd.Series(
        {
            "revenue": round(group["revenue"].sum(), 2),
            "units_per_order": round(group["quantity"].sum() / group["order_id"].nunique(), 2),
            "top_product": best.idxmax(),
            "top_product_share_pct": round(100 * best.max() / best.sum(), 1),
        }
    )


valid.groupby("category")[["order_id", "product_name", "quantity", "revenue"]].apply(describe_category)

,revenue,units_per_order,top_product,top_product_share_pct
category,,,,
Books,70290.25,2.32,Data Systems Design,25.5
Electronics,145420.33,2.25,Noise-Cancelling Headphones,28.9
Home,54476.32,2.22,Air Fryer,21.1
Sports,65399.62,2.19,Tennis Racket,43.5
Toys,37648.07,2.24,Remote Control Car,48.6


## SQL and pandas side by side
| SQL | pandas |
| --- | --- |
| `GROUP BY a` | `df.groupby("a")` |
| `SUM(x) AS total` | `.agg(total=("x", "sum"))` |
| `COUNT(*)` | `("col", "size")` |
| `COUNT(x)` | `("x", "count")` |
| `COUNT(DISTINCT x)` | `("x", "nunique")` |
| `HAVING` | filter the aggregated result, or `.filter(...)` for detail rows |
| `SUM(x) OVER (PARTITION BY a)` | `df.groupby("a")["x"].transform("sum")` |
| `ROW_NUMBER() OVER (PARTITION BY a ORDER BY b)` | sort by `b`, then `groupby("a").cumcount() + 1` |
| `LAG(x) OVER (PARTITION BY a ORDER BY b)` | sort by `b`, then `groupby("a")["x"].shift()` |

## Key takeaways
- Use named aggregation for readable results.
- `agg` reduces, `transform` keeps the shape, `filter` selects groups, `apply` is the fallback.
- `size` counts rows, `count` counts non-missing values.
- Missing keys vanish unless you pass `dropna=False`.

## Practice
1. Revenue and number of orders per sales channel and status.
2. For each country, the customer with the highest spend.
3. Add a column with each order line's share of its **order's** total revenue.
4. Monthly revenue per channel as a table with one column per channel.